# Task 2 — Decision Trees

**Seed:** last 4 digits of student ID = **3039**

In [1]:
import sys

import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__)

3.12.3 2.0.2 2.3.3 1.6.1


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 3039

data = pd.read_csv("data/heart_disease.csv")
feature_names = [c for c in data.columns if c != "disease"]
X, y = data[feature_names].to_numpy(), data["disease"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=SEED
)
print(data.shape, round(y.mean(), 3), X_train.shape, X_test.shape)

(297, 14) 0.461 (207, 13) (90, 13)


## Part 1

### 1. Parent entropy

12 patients: 6 with disease (p=1), 6 without (p=0). Both proportions are 0.5.

H = −(0.5 · log2(0.5) + 0.5 · log2(0.5))
  = −(0.5 · (−1) + 0.5 · (−1))
  = 1.0 bit

### 2. Information gain and root

**age_55_plus**

- `age_55_plus = 1`: 6 patients (5 with disease, 1 without)
  H = -(5/6 · log2(5/6) + 1/6 · log2(1/6)) = 0.650
- `age_55_plus = 0`: 6 patients (1 with disease, 5 without)
  H = -(1/6 · log2(1/6) + 5/6 · log2(5/6)) = 0.650
- Weighted H = (6/12) · 0.650 + (6/12) · 0.650 = 0.650
- **Gain = 1.0 - 0.650 = 0.350**


**exercise_angina**

- `exercise_angina = 1`: 4 patients (all 4 with disease)
  H = 0 (pure node)
- `exercise_angina = 0`: 8 patients (2 with disease, 6 without)
  H = -(2/8 · log2(2/8) + 6/8 · log2(6/8)) = 0.811
- Weighted H = (4/12) · 0 + (8/12) · 0.811 = 0.541
- **Gain = 1.0 - 0.541 = 0.459**


**male** 

- `male = 1`: 6 patients (3 with disease, 3 without)
- `male = 0`: 6 patients (3 with disease, 3 without)
- Weighted H = (6/12) · 1.0 + (6/12) · 1.0 = 1.0
- **Gain = 1.0- 1.0 = 0.000**


**high_bp**

- `high_bp = 1`: 6 patients (4 with disease, 2 without)
  H = -(4/6 · log2(4/6) + 2/6 · log2(2/6)) = 0.918
- `high_bp = 0`: 6 patients (2 with disease, 4 without)
  H = 0.918 (symmetric)
- Weighted H = 0.918
- **Gain = 1.0 - 0.918 = 0.082**


**Summary of gains:**

| Feature | Gain |
| --- | --- |
| age_55_plus | 0.350 |
| exercise_angina | **0.459** |
| male | 0.000 |
| high_bp | 0.082 |

**Root:** `exercise_angina` - it has the highest information gain (0.459)

### 3. One level down

After splitting on exercise_angina, the branch exercise_angina = 1 is pure because all 4 patients have disease.

The impuren branch is exercise_angina =0. It contains 8 patients: 2 with disease and 6 without.

H = -(2/8 · log2(2/8) + 6/8 · log2(6/8)) = 0.811


**age_55_plus**
- `age_55_plus = 1`: 2 patients (1 with disease, 1 without)
- `age_55_plus = 1`: 6 patients (1 with disease, 5 without)
  H = -(1/6 · log2(1/6) + 5/6 · log2(5/6)) = 0.650
- Weighted H = (2/8) · 1.000 + (6/8) · 0.650 = 0.738
- **Gain = 0.811 - 0.738 = 0.074**


**male**

- `male = 1`: 3 patients (0 with disease, 3 without) -> H = 0.000 (pure)
- `male = 0`: 5 patients (2with disease, 3 without)
  H = -(2/5 · log2(2/5) + 3/5 · log2(3/5)) = 0.971
- Weighted H = (3/8) · 0.000 + (5/8) · 0.971 = 0.607
- **Gain = 0.811 - 0.607 = 0.204**


**high_bp**

- `high_bp = 1`: 3 patients (1 with disease, 2 without) -> H = 0.918
- `high_bp = 0`: 5 patients (1 with disease, 4 without) -> H = 0.722
- Weithed H = (3/8) · 0.918 + (5/8) · 0.722 = 0.796
- **Gain  = 0.811 - 0.796 = 0.016**


**Summary - gains in branch vas gains at root:**

| Feature | Gain in branch | Gain at root |
| --- | --- | --- |
| age_55_plus | 0.074 | 0.350 |
| **male** | **0.204** | **0.000** |
| high_bp | 0.016 | 0.082 |


**Chosen second featre:** `male` - highest gain in this branch (0.204).

At the root, male had an information gain of 0, but inside the exercise_angina = 0 branch its gain is 0.204. A feature can be useless at the root but useful inside a branch because after the first split we are working with a different subset of the data.

In [ ]:
#Task 2
#entropy
import numpy as np
def entropy(y):
    """Entropy in bits of an array of integer class labels."""
    if len(y) == 0:
        return 0.0
    _, counts = np.unique(y, return_counts=True)
    p = counts / len(y)
    return -np.sum(p * np.log2(p))

print("Pure (all 1s):", entropy(np.array([1, 1, 1, 1])))
print("Pure (all 0s):", entropy(np.array([0, 0, 0, 0])))
print("50/50:", entropy(np.array([0, 0, 1, 1])))
print("Toy table:", entropy(toy["disease"].values))

Pure (all 1s): -0.0
Pure (all 0s): -0.0
50/50: 1.0
Toy table: 1.0


In [ ]:
#information gain
def information_gain(y, left):
    """Entropy of y minus the size-weighted entropy of y[left] and y[~left]; left is a boolean mask."""
    n = len(y)
    n_left = left.sum()
    n_right = n - n_left
    if n_left == 0 or n_right == 0:
        return 0.0
    return entropy(y) - (n_left / n) * entropy(y[left]) - (n_right / n) * entropy(y[~left])


y_toy = toy["disease"].values
for col in ["age_55_plus", "exercise_angina", "male", "high_bp"]:
    left = toy[col].values == 1
    gain = information_gain(y_toy, left)
    print(f"{col}: gain = {gain:.4f}")

age_55_plus: gain = 0.3500
exercise_angina: gain = 0.4591
male: gain = 0.0000
high_bp: gain = 0.0817


In [10]:
#best split
def best_split(X, y, min_samples_leaf=1):
    """Return (feature_index, threshold, gain) of the best split, or None if no valid split exists."""
    best_gain = -1.0
    best_feature = None
    best_threshold = None
    n_features = X.shape[1]

    for f in range(n_features):
        values = np.unique(X[:, f])
        if len(values) < 2:
            continue
        thresholds = (values[:-1] + values[1:]) / 2
        for t in thresholds:
            left = X[:, f] <= t
            n_left = left.sum()
            n_right = len(y) - n_left
            if n_left < min_samples_leaf or n_right < min_samples_leaf:
                continue
            gain = information_gain(y, left)
            if gain > best_gain:
                best_gain = gain
                best_feature = f
                best_threshold = t

    if best_feature is None:
        return None
    return best_feature, best_threshold, best_gain


toy_X = toy[["age_55_plus", "exercise_angina", "male", "high_bp"]].values
toy_y = toy["disease"].values
result = best_split(toy_X, toy_y)
print("Toy best split:", result)

Toy best split: (1, np.float64(0.5), np.float64(0.4591479170272448))


In [11]:
result_real = best_split(X_train, y_train)
print("Real best split:", result_real)
print("Feature:", feature_names[result_real[0]])
print("Threshold:", result_real[1])
print("Gain:", result_real[2])

Real best split: (11, np.float64(0.5), np.float64(0.24114650717204555))
Feature: ca
Threshold: 0.5
Gain: 0.24114650717204555


In [12]:
from sklearn.tree import DecisionTreeClassifier

ref = DecisionTreeClassifier(criterion="entropy", max_depth=1, random_state=SEED).fit(X_train, y_train)
print(f"sklearn: {feature_names[ref.tree_.feature[0]]}, threshold = {ref.tree_.threshold[0]:.4f}")

sklearn: ca, threshold = 0.5000


In [13]:
#build tree
def build_tree(X, y, max_depth=None, min_samples_leaf=1, depth=0):
    """Grow a tree recursively and return it as nested dicts."""

    if len(np.unique(y)) == 1:
        return {"leaf": True, "prediction": int(y[0]), "n": len(y)}
    
    if max_depth is not None and depth >= max_depth:
        counts = np.bincount(y)
        return {"leaf": True, "prediction": int(np.argmax(counts)), "n": len(y)}
    
    split = best_split(X, y, min_samples_leaf)
    if split is None:
        counts = np.bincount(y)
        return {"leaf": True, "prediction": int(np.argmax(counts)), "n": len(y)}
    
    
    
    feature, threshold, gain = split
    left_mask = X[:, feature] <= threshold
    left = build_tree(X[left_mask], y[left_mask], max_depth, min_samples_leaf, depth + 1)
    right = build_tree(X[~left_mask], y[~left_mask], max_depth, min_samples_leaf, depth + 1)
    
    return {
        "leaf": False,
        "feature": feature,
        "threshold": threshold,
        "gain": gain,
        "n": len(y),
        "left": left,
        "right": right,
    }

In [14]:
#predict tree
def predict_tree(tree, X):
    """Return one predicted label per row of X."""
    predictions = []
    for row in X:
        node = tree
        while not node["leaf"]:
            if row[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        predictions.append(node["prediction"])
    return np.array(predictions)

In [15]:
toy_X = toy[["age_55_plus", "exercise_angina", "male", "high_bp"]].values
toy_y = toy["disease"].values

tree_toy = build_tree(toy_X, toy_y, max_depth=3)
print("Tree built on toy")
print("Root feature:", tree_toy["feature"], "=", ["age_55_plus", "exercise_angina", "male", "high_bp"][tree_toy["feature"]])
print("Root threshold:", tree_toy["threshold"])

preds_toy = predict_tree(tree_toy, toy_X)
print("Predictions:", preds_toy)
print("Actual:     ", toy_y)
print("Accuracy on toy:", (preds_toy == toy_y).mean())


Tree built on toy
Root feature: 1 = exercise_angina
Root threshold: 0.5
Predictions: [1 1 1 1 1 0 0 0 0 0 0 0]
Actual:      [1 1 1 1 1 1 0 0 0 0 0 0]
Accuracy on toy: 0.9166666666666666


In [16]:
tree_real = build_tree(X_train, y_train, max_depth=3)
preds_real = predict_tree(tree_real, X_train)
print("Train accuracy (my tree, depth=3):", (preds_real == y_train).mean())

preds_test = predict_tree(tree_real, X_test)
print("Test accuracy (my tree, depth=3):", (preds_test == y_test).mean())

Train accuracy (my tree, depth=3): 0.8405797101449275
Test accuracy (my tree, depth=3): 0.8111111111111111


In [17]:
from sklearn.tree import DecisionTreeClassifier

print(f"{'Depth':<10} {'Agreement':<12}")
print("-" * 25)

for depth in [1, 2, 3, 4, 5, None]:
    my_tree = build_tree(X_train, y_train, max_depth=depth)
    my_preds = predict_tree(my_tree, X_test)
    
    sk_tree = DecisionTreeClassifier(
        criterion="entropy", max_depth=depth, random_state=SEED
    ).fit(X_train, y_train)
    sk_preds = sk_tree.predict(X_test)
    
    agreement = (my_preds == sk_preds).mean()
    print(f"{str(depth):<10} {agreement:<12.4f}")

Depth      Agreement   
-------------------------
1          1.0000      
2          1.0000      
3          1.0000      
4          1.0000      
5          0.9667      
None       0.9667      
